# Experiment: Neuro-Inspired Probabilistic Risk Prototype

This notebook explores the local research prototype for **neurological risk stratification with calibrated uncertainty**.

Safety notes:
- Research prototype only.
- Not a medical diagnosis tool.
- Softmax confidence does not automatically imply clinical certainty.
- Uncertainty estimates should be displayed together with every prediction.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import IFrame, Image, Markdown, display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

REPORT_PATH = PROJECT_ROOT / "outputs" / "neuro_risk" / "report.json"
JSVIZ_PAYLOAD_PATH = PROJECT_ROOT / "jsviz" / "public" / "latest_inference.json"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "neuro_risk" / "figures"
INTERACTIVE_HTML = PROJECT_ROOT / "outputs" / "neuro_risk" / "interactive_uncertainty.html"

if not REPORT_PATH.exists():
    raise FileNotFoundError(
        "Run `python scripts/run_neuro_risk_mvp.py --epochs 6 --mc-samples 8 --device cpu` before opening this notebook."
    )

report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
payload = json.loads(JSVIZ_PAYLOAD_PATH.read_text(encoding="utf-8"))
display(Markdown(f"Loaded report from `{REPORT_PATH}`"))


## Overview

- Modalities in the MVP: synthetic tabular biomarkers + synthetic temporal neural signals.
- Backbone: deep tabular MLP plus temporal 1D convolution branch.
- Uncertainty: Monte Carlo Dropout with mean predictive probability, class variance, predictive entropy and mutual information.
- Calibration: post-hoc temperature scaling fitted on the validation split.


In [ ]:
metrics_table = pd.DataFrame(
    {
        name: {
            "accuracy": values["accuracy"],
            "f1_macro": values["f1_macro"],
            "auroc_ovr": values["auroc_ovr"],
            "ece": values["ece"],
            "nll": values["nll"],
            "mean_confidence": values["mean_confidence"],
            "mean_entropy": values["mean_entropy"],
        }
        for name, values in report["metrics"].items()
    }
).T
metrics_table


## Calibration summary

Temperature scaling adjusts the sharpness of logits on the validation split without changing the classifier weights.


In [ ]:
pd.DataFrame(
    [
        {
            "temperature": report["temperature"],
            "nll_before": report["temperature_nll_before"],
            "nll_after": report["temperature_nll_after"],
            "caution": "temperature scaling improves reliability but does not replace domain validation",
        }
    ]
)


In [ ]:
figure_paths = [
    FIGURES_DIR / "confusion_matrix.png",
    FIGURES_DIR / "logit_distribution.png",
    FIGURES_DIR / "softmax_probability_heatmap.png",
    FIGURES_DIR / "mc_uncertainty_panels.png",
    FIGURES_DIR / "confidence_distribution.png",
    FIGURES_DIR / "reliability_before_after.png",
]

for path in figure_paths:
    display(Markdown(f"### {path.name}"))
    display(Image(filename=str(path)))


In [ ]:
uncertainty_table = pd.DataFrame(payload["uncertain_examples"])
uncertainty_table


In [ ]:
IFrame(src=str(INTERACTIVE_HTML.relative_to(PROJECT_ROOT)), width="100%", height=640)


## Next steps

- Replace the synthetic generator with real multimodal data loaders.
- Add subject-level evaluation and leakage checks.
- Compare MC Dropout with deep ensembles or evidential approaches.
- Expand the fusion block for EEG + tabular + biomarker inputs.
- Keep the safety disclaimer and uncertainty display in every downstream interface.
